# 06 — Offline video → MMPose → Joint/Motion ST-GCN

This notebook performs inference only. It does not train or modify either frozen action model. The production path is person detection → MMPose COCO-17 → the exact MMAction2 validation transforms → frozen Joint and Joint Motion ST-GCN → fixed 0.5/0.5 fusion → overlapping-window EMA → annotated MP4.

Kaggle requirements: GPU enabled, Internet enabled for official detector/pose weights, both frozen `.pth` files attached, one demo MP4, and a five-row domain-gap manifest plus its videos for the final acceptance cells.


In [ ]:
# Project setup — edit only the optional input overrides when auto-discovery is ambiguous.
from pathlib import Path
import os, sys

REPO_URL = 'https://github.com/mzuyyy/Human-action-recognition.git'
PROJECT_DIR = Path('/kaggle/working/ntu-action-recognition')
MMACTION2_DIR = Path('/kaggle/working/mmaction2')
DEMO_VIDEO = None          # e.g. '/kaggle/input/my-videos/clapping.mp4'
DOMAIN_GAP_MANIFEST = None # e.g. '/kaggle/input/my-videos/domain_gap_manifest.csv'

if not PROJECT_DIR.exists():
    !git clone {REPO_URL} {PROJECT_DIR}
os.chdir(PROJECT_DIR)
for relative in (
    'artifacts/checkpoints', 'artifacts/inference', 'artifacts/demo',
    'assets', 'scripts', 'src/inference'):
    (PROJECT_DIR / relative).mkdir(parents=True, exist_ok=True)
print('project:', PROJECT_DIR)


In [ ]:
%%bash
# Pinned inference stack. Do not reinstall Kaggle's PyTorch/TorchVision.
set -euo pipefail
python -m pip --version
python - <<'PY'
import torch, torchvision
print('preserving torch:', torch.__version__)
print('preserving torchvision:', torchvision.__version__)
PY

# Python 3.12 has no suitable prebuilt full-MMCV wheel for this stack. Both
# ST-GCN and the selected MMPose MobileNetV2 model work with mmcv-lite.
python -m pip uninstall -q -y mmcv mmcv-lite >/dev/null 2>&1 || true
python -m pip install -q --only-binary=mmcv-lite \
  'importlib-metadata' 'mmengine>=0.9.0,<1.0.0' 'mmcv-lite==2.1.0'

MMACTION2_SRC=/kaggle/working/mmaction2
if [ ! -d "${MMACTION2_SRC}/.git" ]; then
  git -c advice.detachedHead=false clone --branch v1.2.0 --depth 1 \
    https://github.com/open-mmlab/mmaction2.git "${MMACTION2_SRC}"
else
  git -C "${MMACTION2_SRC}" fetch -q --depth 1 origin tag v1.2.0 || true
  git -c advice.detachedHead=false -C "${MMACTION2_SRC}" checkout -q --detach v1.2.0
fi
python -m pip uninstall -q -y mmaction2 >/dev/null 2>&1 || true
python -m pip install -q --no-deps -e "${MMACTION2_SRC}"

# Disable only MMAction2's unused optional ViNLU registry. Kaggle's current
# Transformers removed an API that this old optional module imports.
python - <<'PY'
from pathlib import Path
path = Path('/kaggle/working/mmaction2/mmaction/utils/dependency.py')
text = path.read_text()
old = "WITH_MULTIMODAL = all(\n    satisfy_requirement(item) for item in ['transformers>=4.28.0'])"
new = "# Disabled for skeleton-only inference.\nWITH_MULTIMODAL = False"
if old in text:
    path.write_text(text.replace(old, new))
elif new not in text:
    raise RuntimeError(f'cannot disable unused multimodal imports in {path}')
PY

# MMPose is installed without dependency resolution so pip cannot replace the
# working mmcv-lite/PyTorch stack. Kaggle already supplies NumPy, SciPy, OpenCV,
# Pillow, Matplotlib, PyTorch, and TorchVision.
python -m pip install -q --upgrade-strategy only-if-needed \
  json-tricks munkres 'xtcocotools>=1.12'
python -m pip install -q --no-deps 'mmpose==1.3.2'

# MMPose 1.3.2 eagerly registers two unrelated heads: EDPose needs compiled
# mmcv.ops and RTMO needs MMDetection. The selected MobileNetV2+HeatmapHead
# uses neither. Disable only those optional registries, then import the full
# remaining MMPose model registry as the acceptance test.
python /kaggle/working/ntu-action-recognition/scripts/prepare_mmpose_lite.py

python - <<'PY'
from pathlib import Path
from importlib.metadata import version
import mmcv, mmengine, mmaction, mmpose, torch, torchvision
from mmpose.apis import inference_topdown, init_model

assert version('mmcv-lite') == '2.1.0'
assert mmcv.__version__ == '2.1.0'
assert mmaction.__version__ == '1.2.0'
assert mmpose.__version__ == '1.3.2'
assert torch.cuda.is_available(), 'enable a Kaggle GPU accelerator'
print('torch:', torch.__version__)
print('torchvision:', torchvision.__version__)
print('mmengine:', mmengine.__version__)
print('mmcv-lite:', mmcv.__version__)
print('mmaction:', mmaction.__version__, 'from', Path(mmaction.__file__).resolve())
print('mmpose:', mmpose.__version__, 'from', Path(mmpose.__file__).resolve())
PY


In [ ]:
# Activate editable sources in this already-running kernel and locate frozen models.
import importlib, json, os, shutil, sys, urllib.request

for source in (str(MMACTION2_DIR.resolve()), str(PROJECT_DIR.resolve())):
    if source not in sys.path:
        sys.path.insert(0, source)
importlib.invalidate_caches()
import mmaction, mmcv, mmengine, mmpose, torch, torchvision

RUN_ENV = os.environ.copy()
RUN_ENV['PYTHONPATH'] = os.pathsep.join(
    [str(MMACTION2_DIR), str(PROJECT_DIR), RUN_ENV.get('PYTHONPATH', '')])
RUN_ENV['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

def link_unique_input(filename):
    destination = PROJECT_DIR / 'artifacts/checkpoints' / filename
    if destination.is_file():
        return destination
    hits = sorted(path for path in Path('/kaggle/input').rglob(filename) if path.is_file())
    if len(hits) != 1:
        raise FileNotFoundError(
            f'attach exactly one {filename}; candidates={list(map(str, hits))}')
    destination.unlink(missing_ok=True)
    destination.symlink_to(hits[0].resolve())
    print('checkpoint:', destination, '->', hits[0])
    return destination

JOINT_CKPT = link_unique_input('stgcn_joint_ntu60_xsub_best.pth')
MOTION_CKPT = link_unique_input('stgcn_joint_motion_ntu60_xsub_best.pth')

# Official MMPose COCO sample used only for the 17-joint contract test.
test_image = PROJECT_DIR / 'assets/test_person.jpg'
if not test_image.is_file():
    attached = sorted(Path('/kaggle/input').rglob('test_person.jpg'))
    if attached:
        shutil.copy2(attached[0], test_image)
    else:
        url = ('https://raw.githubusercontent.com/open-mmlab/mmpose/'
               'v1.3.2/tests/data/coco/000000000785.jpg')
        temporary = test_image.with_suffix('.jpg.part')
        urllib.request.urlretrieve(url, temporary)
        temporary.replace(test_image)
print('test image:', test_image)
print('frozen checkpoints ready; no training state will be loaded')


In [ ]:
# Phase 1 — compare code against both resolved validation configs and verify
# official Joint/Motion transforms numerically on a synthetic pose sequence.
import subprocess
subprocess.run(
    [sys.executable, 'scripts/verify_inference_preprocessing.py'],
    cwd=PROJECT_DIR, env=RUN_ENV, check=True)


In [ ]:
# Phases 2–3 — one official lightweight COCO-17 pose inference.
subprocess.run(
    [sys.executable, 'scripts/test_mmpose_image.py', str(test_image),
     '--output', 'artifacts/inference/test_pose.jpg'],
    cwd=PROJECT_DIR, env=RUN_ENV, check=True)
from IPython.display import Image, display
display(Image(filename=str(PROJECT_DIR / 'artifacts/inference/test_pose.jpg')))


In [ ]:
# Resolve the demo clip. Prefer an explicit path, then demo_input.mp4, then
# the only attached MP4. If several exist, set DEMO_VIDEO in the first cell.
def resolve_demo_video(value):
    if value is not None:
        path = Path(value).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    preferred = sorted(Path('/kaggle/input').rglob('demo_input.mp4'))
    if len(preferred) == 1:
        return preferred[0].resolve()
    candidates = sorted(path.resolve() for path in Path('/kaggle/input').rglob('*.mp4'))
    if len(candidates) != 1:
        raise FileNotFoundError(
            'set DEMO_VIDEO to one attached MP4; candidates=' +
            json.dumps(list(map(str, candidates)), indent=2))
    return candidates[0]

DEMO_VIDEO_PATH = resolve_demo_video(DEMO_VIDEO)
print('demo video:', DEMO_VIDEO_PATH)


In [ ]:
# Phases 4–6 — extract one dominant pose sequence, then prove Joint-only
# action inference works before constructing fusion.
POSE_FILE = PROJECT_DIR / 'artifacts/inference/example_pose.pkl'
subprocess.run(
    [sys.executable, 'scripts/extract_video_pose.py', str(DEMO_VIDEO_PATH),
     '--output', str(POSE_FILE)],
    cwd=PROJECT_DIR, env=RUN_ENV, check=True)
subprocess.run(
    [sys.executable, 'scripts/infer_action_joint.py', str(POSE_FILE),
     '--checkpoint', str(JOINT_CKPT)],
    cwd=PROJECT_DIR, env=RUN_ENV, check=True)


In [ ]:
# Phases 7–11 and 13 — official Joint Motion, frozen 50/50 fusion,
# overlapping 100-frame/50-stride windows, EMA, profiling, annotated MP4.
DEMO_OUTPUT = PROJECT_DIR / 'artifacts/demo/demo_output.mp4'
subprocess.run(
    [sys.executable, 'scripts/demo_video.py', str(DEMO_VIDEO_PATH),
     '--pose-file', str(POSE_FILE), '--output', str(DEMO_OUTPUT),
     '--joint-checkpoint', str(JOINT_CKPT),
     '--motion-checkpoint', str(MOTION_CKPT),
     '--show-stream-scores'],
    cwd=PROJECT_DIR, env=RUN_ENV, check=True)
from IPython.display import Video, display
display(Video(str(DEMO_OUTPUT), embed=False))


In [ ]:
# Phase 12 — required qualitative domain-gap set (at least five clips).
# CSV columns: video_path,expected_action. Relative video paths are resolved
# beside the manifest. Expected actions must exactly match an NTU60 class name.
def resolve_domain_manifest(value):
    if value is not None:
        path = Path(value).expanduser().resolve()
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    hits = sorted(Path('/kaggle/input').rglob('domain_gap_manifest.csv'))
    if len(hits) != 1:
        raise FileNotFoundError(
            'attach exactly one domain_gap_manifest.csv plus at least five '
            f'labeled videos; candidates={list(map(str, hits))}')
    return hits[0].resolve()

DOMAIN_MANIFEST_PATH = resolve_domain_manifest(DOMAIN_GAP_MANIFEST)
subprocess.run(
    [sys.executable, 'scripts/test_domain_gap.py', str(DOMAIN_MANIFEST_PATH),
     '--joint-checkpoint', str(JOINT_CKPT),
     '--motion-checkpoint', str(MOTION_CKPT)],
    cwd=PROJECT_DIR, env=RUN_ENV, check=True)


In [ ]:
# Final acceptance report. This refuses to run without a real demo, measured
# latency, verified preprocessing, and at least five domain-gap rows.
subprocess.run(
    [sys.executable, 'scripts/report_video_inference.py'],
    cwd=PROJECT_DIR, env=RUN_ENV, check=True)
print((PROJECT_DIR / 'artifacts/inference/video_inference_report.md').read_text())
print('STOP — no training, ONNX, TensorRT, ByteTrack, or other experiment started.')
